# Phase 06B.00 — Taxonomy slice analysis

Runs only after the schema-v2 human taxonomy gate.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"src").is_dir()), None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))
from roadbuddy_common import save_json
from phase06a_common import sha256_file,sha256_json
from phase06b_common import LOCKED_BASELINE_WINNER,build_novelty_decision_evidence,load_phase06a_predictions,taxonomy_slice_metrics,validate_locked_baseline,validate_taxonomy_frame,validate_taxonomy_manifest
OUT=ROOT/"outputs/phase06b/taxonomy_slice_analysis"; OUT.mkdir(parents=True,exist_ok=True)
TD=ROOT/"outputs/phase06a/question_taxonomy"; TP=TD/"taxonomy_frozen.csv"; TM=TD/"taxonomy_manifest.json"; IDS=ROOT/"data/splits/phase01/validation_sample_ids.json"
baseline=validate_locked_baseline(ROOT)


In [ ]:
missing=[str(p.relative_to(ROOT)) for p in [TP,TM] if not p.is_file()]
if missing:
    save_json(OUT/"PHASE06B_00_STATUS.json",{"status":"awaiting_annotation","schema_version":2,"missing":missing,"required":["annotator_1.csv","annotator_2.csv","taxonomy_adjudicated.csv","taxonomy_frozen.csv","taxonomy_manifest.json"]})
    raise RuntimeError("Complete the human taxonomy gate")
manifest=validate_taxonomy_manifest(json.loads(TM.read_text()),taxonomy_path=TP,validation_ids_path=IDS)
if manifest["status"]!="complete": raise RuntimeError("Taxonomy requires re-annotation")
ids=json.loads(IDS.read_text()); taxonomy=validate_taxonomy_frame(pd.read_csv(TP),ids)


In [ ]:
predictions=load_phase06a_predictions(ROOT,ids)
slices=taxonomy_slice_metrics(predictions,taxonomy,min_rows=30,min_groups=15); slices.to_csv(OUT/"taxonomy_slice_metrics.csv",index=False)
evidence=build_novelty_decision_evidence(predictions[LOCKED_BASELINE_WINNER],taxonomy,min_rows=30,min_groups=15,min_error_share=.30,min_error_share_gap=.10)
save_json(OUT/"novelty_track_evidence.json",evidence)
save_json(OUT/"PHASE06B_00_STATUS.json",{"status":"complete","taxonomy_sha256":sha256_file(TP),"evidence_sha256":sha256_json(evidence),"recommendation":evidence["recommendation"],"human_decision_required":True})
slices
